In this notebook we pull the [stem.MAP model](https://huggingface.co/sfisch/stem.MAP_AutoGluon/blob/main/README.md) and the **full feature matrix**, which contains data for >96 million protein pairs, from our [HuggingFace](https://huggingface.co/datasets/sfisch/stem.MAP) and make predictions.

In [1]:
# AutoGluon Version 0.8.2 is needed to use with our model
# Others can be used if the user trains their own model
from autogluon.tabular import TabularPredictor

#sfisch6: we pull the full feature matrix from huggingface using the datasets module
from datasets import load_dataset

#sfisch6: we use this module to pull our trained model from huggingface
from huggingface_hub import snapshot_download
import polars as pl

In [2]:
#sfisch6: load in model 
model_dir = snapshot_download(repo_id="sfisch/stem.MAP_AutoGluon")
predictor = TabularPredictor.load(f"{model_dir}/stem.MAP_AutoGluon_model_20240731")

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

/home/jupyter-sfisch6_2025/.local/lib/python3.9/site-packages/autogluon/common/utils/utils.py:74: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
#sfisch6: temporary work-around while repo is still private

from getpass import getpass

token = getpass("Hugging Face token: ")

Hugging Face token: ········


In [6]:
#sfisch6: while using token

#sfisch6: load full feature matrix
# note that to keep this example light, we will use polars to
# pull the first 500 rows only 
# to pull entire feature matrix for prediction
# uncomment the next code block to download w/ huggingface_hub package
full_fpath = (
    "hf://datasets/sfisch/stem.MAP/full/"
    "stem.MAP_all_pairs_full_feature_set_20240815.parquet")

full_featmat = pl.read_parquet(
    full_fpath,
    n_rows=500,
    storage_options={"token": token}).to_pandas()

In [7]:
#sfisch6: load full feature matrix
# note that to keep this example light, we will use polars to
# pull the first 500 rows only 
# to pull entire feature matrix for prediction
# uncomment the next code block to download w/ huggingface_hub package
#full_fpath = ("hf://datasets/sfisch/stem.MAP/full/"
#    "stem.MAP_all_pairs_full_feature_set_20240815.parquet")

#sfisch6: read in full fmat using polars to grab only the 1st 500 rows
#full_featmat = pl.read_parquet(full_fpath, n_rows=500).to_pandas() 

In [8]:
# sfisch6: uncomment this block for full feature matrix import #
#from huggingface_hub import hf_hub_download

#full_fpath = hf_hub_download(repo_id="sfisch/stem.MAP", 
#                             filename='full/stem.MAP_all_pairs_full_feature_set_20240815.parquet',
#                             repo_type='dataset')

# sfisch6: Note that this make take a long time as it is a very large matrix #
#full_featmat = pd.read_parquet(full_fpath) 

In [11]:
#sfisch6: as with testing and training, we removed the ID column
full_trim = full_featmat.drop(['ID'], axis=1)

In [12]:
full_trim.head()

,pearsonR_MES_IEX_mixedbed_20180212,pearsonR_Mallam_MES_SEC_RNAse,pearsonR_Mallam_MES_control,humap3_pearsonR_MES_IEX_mixedbed_20180212,humap3_pearsonR_Mallam_MES_SEC_RNAse,humap3_pearsonR_Mallam_MES_control
0,-0.0415,0.0000,0.0000,0.000006,0.000006,0.000001
1,0.0000,0.0072,-0.0499,0.000008,0.000003,0.000006
2,0.0000,-0.0374,-0.0451,0.000008,0.000006,0.000006
3,0.0000,-0.0440,-0.0605,0.000004,0.000004,0.000009
4,0.0000,0.0000,0.0000,0.000005,0.000005,0.000003


In [13]:
full_trim.shape

(500, 6)

In [14]:
#sfisch6: generate predictions
y_pred = predictor.predict(full_trim)
print("Predictions:  \n", y_pred)

Predictions:  
 0     -1
1     -1
2     -1
3     -1
4     -1
      ..
495   -1
496   -1
497   -1
498   -1
499   -1
Name: IntAct_label, Length: 500, dtype: int64


In [15]:
#sfisch6: generate probabilities for the predictions
pred_df_probs = predictor.predict_proba(full_trim)

In [16]:
#sfisch6: looking at the probability of an interaction (1)
pred_df_probs.sort_values(1)

,-1,1
51,0.997837,0.002163
481,0.997837,0.002163
297,0.997837,0.002163
224,0.997837,0.002163
162,0.997837,0.002163
...,...,...
327,0.527071,0.472929
207,0.500500,0.499500
43,0.499390,0.500610
29,0.369173,0.630827


In [17]:
full_featmat['pred_prob'] = pred_df_probs[1]

In [18]:
full_featmat.head()

,ID,pearsonR_MES_IEX_mixedbed_20180212,pearsonR_Mallam_MES_SEC_RNAse,pearsonR_Mallam_MES_control,humap3_pearsonR_MES_IEX_mixedbed_20180212,humap3_pearsonR_Mallam_MES_SEC_RNAse,humap3_pearsonR_Mallam_MES_control,pred_prob
0,A0A087WNM5 A0A087WNL7,-0.0415,0.0000,0.0000,0.000006,0.000006,0.000001,0.004521
1,Q91YE9 Q8BFR6,0.0000,0.0072,-0.0499,0.000008,0.000003,0.000006,0.005348
2,Q91W39 Q8BFR6,0.0000,-0.0374,-0.0451,0.000008,0.000006,0.000006,0.002928
3,Q8BFT2 Q8CC27,0.0000,-0.0440,-0.0605,0.000004,0.000004,0.000009,0.003091
4,Q8BFR6 Q9CRY7,0.0000,0.0000,0.0000,0.000005,0.000005,0.000003,0.005551


In [20]:
(full_featmat
 .sort_values("pred_prob", ascending=False)[['ID','pred_prob']]
 .to_csv(
    "stem.MAP_unlabeledpredictions_first_500pairs.pairsWprob", 
     index=False, 
     header=False, 
     sep='\t')
)